Full name: Elsa Ingrid Daniela Erkfeldt
Civil nummer: 200309021228
LLM used: ChatGPT was used to get a deeper explanation for some slides in the PPT

In [ ]:
!pip install tensorflow
!pip install scikit-learn scikit-image

In [4]:
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras.datasets import mnist, cifar10
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras import callbacks
tf.random.set_seed(42)
#Distance computation
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from collections import Counter

Download the data sets and normalise the data
For MNIST:
    - Use min-max scaling, so that the values are in range[0,1]
For CIFAR-10:
    -Use Z score normalisation (mean=0, stdev=1)

In [5]:

(x_raw, y_raw), (x_test_raw, y_test)= mnist.load_data()

val_size = int(x_raw.shape[0] * 0.3)
rng = np.random.default_rng(42)  # seed for reproducibility
idx = np.arange(x_raw.shape[0])
rng.shuffle(idx)

val_idx = idx[:val_size]
train_idx = idx[val_size:]

(x_train_raw, y_train), (x_val_raw, y_val) = (x_raw[train_idx], y_raw[train_idx]), (x_raw[val_idx], y_raw[val_idx])


(x_raw_cif, y_cif), (x_test_cif_raw, y_test_cif) = cifar10.load_data()

val_size_cif = int(x_raw_cif.shape[0]*0.3)
idx_cif = np.arange(x_raw_cif.shape[0])
rng.shuffle(idx_cif)

val_idx_cif = idx_cif[:val_size_cif]
train_idx_cif = idx_cif[val_size_cif:]

(x_train_cif_raw, y_train_cif), (x_val_cif_raw, y_val_cif) = (x_raw_cif[train_idx_cif], y_cif[train_idx_cif]), (x_raw_cif[val_idx_cif], y_cif[val_idx_cif])

x_train_cif_type = x_train_cif_raw.astype("float32")
x_test_cif_type = x_test_cif_raw.astype("float32")
x_validation_cif_type = x_val_cif_raw.astype("float32")

In [6]:
x_train, x_test, x_val= x_train_raw/255.0, x_test_raw/255.0, x_val_raw/255.0

def z_score_normalisation(train, test, validation):
    mean = np.mean(train, axis= (0,1,2), keepdims=True)
    std = np.std(train, axis= (0,1,2), keepdims=True)

    return (train - mean) / std, (test - mean) / std, (validation -mean) /std


x_train_cif, x_test_cif, x_val_cif = z_score_normalisation(x_train_cif_type, x_test_cif_type, x_validation_cif_type)



MNIST.
Train (with Adam optimizer) a CNN with:
    -First layer: Convolution layer wtih 16-64 filters (3x3), ReLu activation and batch normalisation
    -Second layer: Max pooling (2x2)
    -Third layer: Convolution layer with 32-128 filters (3x3), ReLu activation and batch normalisation
    -Fourth layer: max pooling (2x2)
    -Fifth layer: (after flattening the output from fourth later): A fully connected ("dense") layer with 128 neurons, ReLu activation (and dropout during training with suitable dropout range (somewhere in [0.2, 0.5]))
    -Sixth layer: a layer with 10 neurons (the number of classes) and a softmax activation function

In [7]:

model = models.Sequential([
    layers.Input(shape = (28,28,1)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3)),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted = model.fit(x_train, y_train, epochs = 10, batch_size = 64, validation_data = (x_val, y_val))

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy: ", test_acc)


Epoch 1/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.8339 - loss: 0.5199 - val_accuracy: 0.9761 - val_loss: 0.0818
Epoch 2/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9676 - loss: 0.1057 - val_accuracy: 0.9838 - val_loss: 0.0562
Epoch 3/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9769 - loss: 0.0756 - val_accuracy: 0.9835 - val_loss: 0.0575
Epoch 4/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9810 - loss: 0.0609 - val_accuracy: 0.9872 - val_loss: 0.0468
Epoch 5/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9831 - loss: 0.0542 - val_accuracy: 0.9873 - val_loss: 0.0436
Epoch 6/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.9852 - loss: 0.0439 - val_accuracy: 0.9875 - val_loss: 0.0465
Epoch 7/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9862 - loss: 0.0387 - val_accuracy: 0.9853 - val_loss: 0.0526
Epoch 8/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9883 - loss: 0.0375 - val_accuracy: 

In [8]:

model_cif = models.Sequential([
    layers.Input(shape = (32,32,3)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Conv2D(filters=32, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #6
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_val_cif, y_val_cif))

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Epoch 1/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.3402 - loss: 1.8298 - val_accuracy: 0.5283 - val_loss: 1.3063
Epoch 2/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5104 - loss: 1.3582 - val_accuracy: 0.5922 - val_loss: 1.1288
Epoch 3/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5707 - loss: 1.2001 - val_accuracy: 0.6250 - val_loss: 1.0428
Epoch 4/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6097 - loss: 1.0959 - val_accuracy: 0.6403 - val_loss: 0.9969
Epoch 5/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6369 - loss: 1.0264 - val_accuracy: 0.6587 - val_loss: 0.9567
Epoch 6/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6529 - loss: 0.9721 - val_accuracy: 0.6725 - val_loss: 0.9319
Epoch 7/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6695 - loss: 0.9306 - val_accuracy: 0.6777 - val_loss: 0.9183
Epoch 8/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6845 - loss: 0.8921 - val_accuracy: 

This one had a test Accuracy of 70%. To make it even better the following changes were made:


*   The number of filters for the Conv2D was adjusted so itstarted at 32, but increased for every layer
*   More convolution layers were added before each pooling too get a better representation of all the features before compressing it
* Pooling was used instead of flattten, to avoid it being too sensitive to the exact location of a pixel
* Dropout was added in more locations to avoid overfitting by forcing it to not rely on any specific feature too much.



In [9]:

model_cif2 = models.Sequential([

    layers.Input(shape = (32,32,3)),

    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #2
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #3
    layers.MaxPooling2D((2,2)),

    #4
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #6
    layers.MaxPooling2D((2,2)),

    #7
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #8
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #9
    layers.MaxPooling2D((2,2)),

    #10
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #11
    layers.Dense(10, activation = "softmax")
])

model_cif2.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif2 = model_cif2.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_val_cif, y_val_cif))

test_loss_cif2, test_acc_cif2 = model_cif2.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif2)

Epoch 1/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 28s 29ms/step - accuracy: 0.3013 - loss: 1.9242 - val_accuracy: 0.3629 - val_loss: 1.7432
Epoch 2/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5150 - loss: 1.3442 - val_accuracy: 0.3813 - val_loss: 1.7176
Epoch 3/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.5910 - loss: 1.1629 - val_accuracy: 0.5100 - val_loss: 1.4656
Epoch 4/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6323 - loss: 1.0505 - val_accuracy: 0.5017 - val_loss: 1.4371
Epoch 5/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6597 - loss: 0.9686 - val_accuracy: 0.5585 - val_loss: 1.2654
Epoch 6/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6798 - loss: 0.9205 - val_accuracy: 0.6415 - val_loss: 1.1080
Epoch 7/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7015 - loss: 0.8645 - val_accuracy: 0.5858 - val_loss: 1.2292
Epoch 8/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7143 - loss: 0.8230 - val_acc

Now the test accuracy has increased slightly to 73%, but we will try to improve it again. This time we added:


*   Early stopping to avoid overfitting and to save time (so it doesnt have to continue if it does not improve)
* Learning rate scheduler: Learning rate is decreased if the model stops improving


In [10]:

model_cif = models.Sequential([

    layers.Input(shape = (32,32,3)),

    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.1),

    #2
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.1),

    #3
    layers.MaxPooling2D((2,2)),

    #4
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    #6
    layers.MaxPooling2D((2,2)),

    #7
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    #8
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    #9
    layers.MaxPooling2D((2,2)),

    #10
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #11
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
early_stopping = callbacks.EarlyStopping(monitor = "val_loss", patience = 4, restore_best_weights = True)
learning_rate = callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.5, patience = 3, min_lr = 1e-5)
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 30, batch_size = 64, validation_data = (x_val_cif, y_val_cif), callbacks = [early_stopping, learning_rate])

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Epoch 1/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 27s 29ms/step - accuracy: 0.3355 - loss: 1.8234 - val_accuracy: 0.4660 - val_loss: 1.4478 - learning_rate: 0.0010
Epoch 2/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5614 - loss: 1.2204 - val_accuracy: 0.5335 - val_loss: 1.2906 - learning_rate: 0.0010
Epoch 3/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6274 - loss: 1.0502 - val_accuracy: 0.6397 - val_loss: 1.0343 - learning_rate: 0.0010
Epoch 4/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6708 - loss: 0.9379 - val_accuracy: 0.6634 - val_loss: 0.9602 - learning_rate: 0.0010
Epoch 5/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6987 - loss: 0.8619 - val_accuracy: 0.6808 - val_loss: 0.9163 - learning_rate: 0.0010
Epoch 6/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7237 - loss: 0.7954 - val_accuracy: 0.6865 - val_loss: 0.9113 - learning_rate: 0.0010
Epoch 7/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7461 - loss: 0

This change lead to a huge improvement in the test accuracy, at 83.7%. This model was therefore picked

Implement kNN models for both MNISTA and CIFAR.
Use the standard euclidean distance measure (applied pixel-by picel)
Run each model with different values of k (from 1-25 eg.). Pick the k with the highers validation accuracy.
Use that to obtain testing accuracy

Flatten each image
Scale data (min-max scaling to put all values in [0, 1])
Distance computation: sum distances (squared, if euclidean norm) over all elements, computed pixel by pixel (compare test image pixels with those of image Ii in data set)
sort (together with label) the distances in increasing order
voting: take the majority label among the k-nearest neighbours

In [11]:

def flatten(x):
    return x.reshape(x.shape[0], -1)

flattened_x_train, flattened_x_test, flattened_x_val = flatten(x_train), flatten(x_test), flatten(x_val)

flattened_x_train_CIFAR, flattened_x_test_CIFAR, flattened_x_val_CIFAR = flatten(x_train_cif), flatten(x_test_cif), flatten(x_val_cif)


Functions for KNN. They are however not used in the rest of the code as each time I tried to use it Colab told me that I was out of alla vailable RAM and that my session had crashed

In [12]:
class KNN():
  def __init__(self, k):
    self.k = k
    self.x_train = None
    self.y_train = None

  def fit(self, x_train, y_train):
    self.x_train = x_train
    self.y_train = y_train

  def predict(self, x_test):

    dists = np.sqrt(np.sum(x_test**2, axis = 1)[:, None] +
                    np.sum(self.x_train**2, axis=1)[None, :] -
                    2* x_test @ self.x_train.T)
    knn_index = np.argpartitions(dists, self.k, axis =1)[:, :self.k]

    knn_labels = self.y_train[knn_index]
    predictions = np.array([Counter(row).most_common(1)[0][0] for row in knn_labels])

    return predictions

  def kneighbours(self, x_test_line, n_neighbours):
    dist = np.sqrt(np.sum(x_test_line**2) + np.sum(self.x_train**2, aaxis=1)- 2*(self.x_train@x_test_line))

    index = np.argpartition(dist, n_neighbours)[:n_neighbours]
    return dist[idx], idx


MNIST finding best n_neighbours
And test accuracy for the best one

In [ ]:
best_k, best_acc = None, -1.0
for k in range(1,26):
    knn = KNeighborsClassifier(n_neighbors=k, metric = "euclidean", weights="uniform")

    knn.fit(flattened_x_train, y_train)
    prediction = knn.predict(flattened_x_val)

    acc = accuracy_score(y_val, prediction)
    print("Accuracy: ", acc, " for k = ", k)
    if acc > best_acc:
        best_acc = acc
        best_k = k

#Test with best k
knn = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", weights="uniform")
knn.fit(flattened_x_train, y_train)
test_acc = accuracy_score(y_test, knn.predict(flattened_x_test))
print("Test accuracy: ", test_acc)

The best one was k=1 with a validation accuracy of 97% and a test accuracy of 96.6 %

Doing the same for CIFAR

In [ ]:
best_acc_CIF = 0
best_k_CIF = None
for k in range(1,26):
    knn = KNeighborsClassifier(n_neighbors = k, metric="euclidean", weights = "uniform")

    knn.fit(flattened_x_train_CIFAR, y_train_cif)
    prediction = knn.predict(flattened_x_val_CIFAR)

    acc = accuracy_score(y_val_cif, prediction)
    print("Accuracy: ", acc, " for k = ", k)
    if acc > best_acc_CIF:
        best_acc_CIF = acc
        best_k_CIF = k

#Test with best k
knn_cif = KNeighborsClassifier(n_neighbors=best_k, metric="euclidean", weights="uniform")
knn_cif.fit(flattened_x_train_CIFAR, y_train_cif)
test_acc_cif = accuracy_score(y_test_cif, knn_cif.predict(flattened_x_test_CIFAR))
print("Test accuracy: ", test_acc_cif)

In [ ]:
test_examples = [0, 5, 50, 500, 5000]

k = 4


Here it shows the image and the nearest neighbour for each of a few classified test images

In [ ]:
for idx in test_examples:
  image_used = flattened_x_test_CIFAR[idx].reshape(1, -1)
  distance, indices = knn_cif.kneighbors(flattened_x_train_CIFAR[idx], n_neighbours = k, return_distance= True)
  indices = indices[0]
  distance = distance [0]

  plt.figure()
  plt.subplot(1, 5, 1)
  plt.imshow(x_test_cif_raw[idx])
  plt.title(f"Image, with true label {y_train_cif[idx]}")

  for i in range(indices.shape(1)):
    plt.subplot(1, 5, i+2)
    plt.imshow(x_test_cif_raw[indices[i]])
    plt.title(f"Neighbour num {indices[i]} with distance {distance[i]:.2f}")

  plt.show()

Comparing wrongly predicted test images to images that have the label they are predicted to have, one notices that its common that different types of mammals are predicted to be deers. This may be because the dataset was not trained long enought to be able to distinguish the features that differ between differe mammals, instead learning those that were the same for all of them (fur, two eyes, two ears, four legs, etc.)